![Platform+](images/banner.png)

# Lab 3: Understand Access / RBAC
_Hands-on Snowflake for data practitioners_

## Why can't I see this table? Grant access the right way.

> **Persona track: Data Analyst** — same lab, your angle.

:point_right: **In this lesson, you'll reproduce an access-denied error, understand why it happens, fix it by granting the right privilege, and learn techniques to manage access across multiple roles.**

The scenario: Your coffee roastery has an ORDERS table and a FEEDBACK table. A new analyst role has been set up — but someone forgot to grant access to the feedback data. You'll experience the failure, then fix it.

We'll cover:
1. How access flows in Snowflake: **privilege → role → user**
2. Setting up the lab scenario (tables + partial grants)
3. Reproducing an access-denied error (the #1 new-user question)
4. Granting the missing privilege to fix it
5. Discovering what you can access with SHOW GRANTS
6. Secondary roles — combining access from multiple roles

:books: **By the end you'll be able to:**
- Explain the full grant chain: warehouse → database → schema → table
- Understand why switching roles can make tables "disappear"
- Grant and revoke SELECT on a table (as the object owner)
- Use SHOW GRANTS to audit your own access
- Use secondary roles to combine privileges without switching roles
- Use CoCo as an access troubleshooting partner
- Apply the principle of least privilege

**Time:** ~35 min · **Lab interface:** Workspace Notebook with Cortex Code (CoCo) as your tutor.
*Source data lives in the shared, read-only `PLATPLUS_DB`; anything you create lives in your own database (named after your username).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


## Set your session context

Every Snowflake session operates under a specific role and warehouse. The role determines what you can see and do — the warehouse provides the compute.

> :bulb: **Well-Architected principle:** Never use ACCOUNTADMIN for daily work or object creation. Use the least-privileged role that can accomplish the task.

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};


#### Step 1: Create the tables in your database

In setting up the lab scenario below copies two tables from the shared `PLATPLUS_DB` into your personal database. We do this so **you own the tables** — which means you can grant privileges on them without needing SECURITYADMIN.

**Why this matters:**
- In Snowflake, the **owner** of an object (the role that created it) can grant privileges on it
- By creating these tables under `{{my_role}}`, you become the owner and can control who else accesses them
- This simulates a real scenario where a team owns their data and manages access to it

| Table | What it contains | Who should have access |
|-------|-----------------|------------------------|
| `ORDERS` | Customer order history | Both {{my_role}} and {{my_analyst_role}} |
| `FEEDBACK` | Customer feedback/reviews | {{my_role}} only (initially) — we'll "forget" to grant {{my_analyst_role}} |


#### Step 2: Grant partial access (deliberately incomplete)

The cell below grants `{{my_analyst_role}}` access to the database, schema, and the ORDERS table — but **intentionally omits** the FEEDBACK table. This sets up the access-denied scenario you'll troubleshoot in Section 3.

**Why we grant in layers (warehouse → database → schema → table):**

Snowflake access is hierarchical. You need grants at **every level** to run a query:

```
Warehouse (USAGE)  →  Database (USAGE)  →  Schema (USAGE)  →  Table (SELECT)
```

Missing any link in the chain = failure. No warehouse = can't execute. No database = can't reference objects. No schema = can't see tables. No SELECT = access denied.

---

**Best practice:** When onboarding a new role, always grant the full chain. Use a checklist:
1. USAGE on warehouse (provides compute — without this, nothing runs)
2. USAGE on database
3. USAGE on schema
4. Specific privilege on object (SELECT, INSERT, etc.)

> :bulb: In this lab, `{{my_analyst_role}}` already has USAGE on `PLATPLUS_WH` (granted during environment setup by the platform admin). That's why the cell below only grants the database → schema → table chain.

> :wrench: **Real-world note — two paths to compute:**
>
> | Path | When to use | Example |
> |------|-------------|--------|
> | **Use a shared warehouse** | Most common for analysts. A platform admin grants USAGE on an existing warehouse. | `GRANT USAGE ON WAREHOUSE shared_wh TO ROLE analyst_rl;` |
> | **Create your own warehouse** | When you have `CREATE WAREHOUSE` privilege (common for data engineers). You own it and can grant access to others. | `CREATE WAREHOUSE my_team_wh WAREHOUSE_SIZE='XSMALL'; GRANT USAGE ON WAREHOUSE my_team_wh TO ROLE analyst_rl;` |
>
> In this lab we use the shared `PLATPLUS_WH`. In production, data engineers often create dedicated warehouses sized for their workloads (e.g., a small warehouse for ad-hoc queries, a larger one for heavy pipeline loads).

---

**Use cases for partial access:**
| Scenario | What you grant | What you withhold |
|----------|---------------|-------------------|
| Analyst sees orders but not PII | SELECT on orders | No access to customer_details table |
| Junior dev reads but can't write | SELECT on all tables | No INSERT, UPDATE, DELETE |
| External partner sees summary views | SELECT on views only | No access to base tables |


In [ ]:
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.RBAC_LAB;

## 1 · How access works in Snowflake

Access in Snowflake follows a simple chain:

```
PRIVILEGE  →  ROLE  →  USER
(what)        (who)    (person)
```

- **Privileges** are permissions on objects (SELECT on a table, USAGE on a schema)
- **Roles** collect privileges — they define "what this job function can do"
- **Users** are assigned roles — they inherit whatever privileges those roles hold

**The key rule:** In Role-Based Access Control (RBAC), you grant privileges to **roles**. Users then inherit whatever privileges their assigned roles hold.

**Why?** Because people change roles, leave companies, get promoted. If you grant to users, you have to individually manage every person. If you grant to roles, you manage the role once and assign people to it.

> :bulb: **Real-world context:** "Why can't I see this table?" is the #1 support question from new Snowflake users. The answer is almost always: your current role doesn't have a grant on that object. The data didn't disappear — your role just can't see it.

**The system-defined role hierarchy:**
```
ACCOUNTADMIN
  ├── SECURITYADMIN  (manages grants + roles)
  │     └── USERADMIN  (manages users)
  └── SYSADMIN  (manages warehouses + databases)
        └── custom roles (yours!)
              └── PUBLIC  (everyone)
```

Higher roles inherit privileges from lower roles. A custom role should always be granted to SYSADMIN (or another parent) so admins can manage objects created by that role.

## 2 · Set up the lab scenario

As the owner of your database (`{{my_role}}`), you'll create two tables and grant **partial** access to a second role (`{{my_analyst_role}}`). This simulates a real scenario: an analyst is set up with access to orders data, but someone forgot to include the feedback table.

> :bulb: **Where does `{{my_analyst_role}}` come from?** This role was pre-created by your instructor as part of the lab environment setup. It has warehouse usage (so it can run queries) but **no access to any database objects** by default. This is intentional — it represents a new team member's role before anyone has granted them access to specific data. Your job in this lab is to grant (and troubleshoot) that access.


In [ ]:
-- Copy two tables into your personal database for the demo
CREATE OR REPLACE TABLE {{my_db}}.RBAC_LAB.ORDERS   AS SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS;
CREATE OR REPLACE TABLE {{my_db}}.RBAC_LAB.FEEDBACK AS SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK;

In [ ]:
-- Warehouse USAGE on PLATPLUS_WH was granted to {{my_analyst_role}} during environment setup.
-- Here we grant the remaining links in the chain: database → schema → table
GRANT USAGE  ON DATABASE {{my_db}}                 TO ROLE {{my_analyst_role}};
GRANT USAGE  ON SCHEMA   {{my_db}}.RBAC_LAB        TO ROLE {{my_analyst_role}};
GRANT SELECT ON TABLE    {{my_db}}.RBAC_LAB.ORDERS TO ROLE {{my_analyst_role}};
-- Note: NO grant on FEEDBACK — that's intentional


> :bulb: **What just happened — decoded:**
>
> | SQL Statement | What it means |
> |---|---|
> | `GRANT USAGE ON WAREHOUSE` | Allows the role to use compute (granted during setup on `PLATPLUS_WH`) |
> | `GRANT USAGE ON DATABASE` | Allows the role to "see" the database exists and enter it |
> | `GRANT USAGE ON SCHEMA` | Allows the role to see the schema and list objects inside |
> | `GRANT SELECT ON TABLE` | Allows the role to query (read) data from that specific table |
>
> **You need ALL FOUR** to query a table. Missing any one link in the chain = failure. The warehouse grant was already in place (from environment setup), so we only needed to add the database → schema → table grants here.

## 3 · Switch roles and hit the wall

Now become `{{my_analyst_role}}` — the limited role. You'll run two queries: one that works and one that fails. The data is the same — the only difference is whether the role has a grant.

**Why this matters (data analyst perspective):**
- **This is the #1 frustration analysts face** — "I know the table exists, my colleague can see it, but I get an error." Understanding *why* it happens is the first step to resolving it quickly.
- **Self-diagnosis saves time** — Instead of filing a support ticket every time you hit an access error, you can identify *exactly* what's missing and request the right grant from your admin.
- **Context switching is normal** — In many organizations, analysts have multiple roles for different projects or data sensitivity levels. Understanding role-switching is a daily skill.


In [ ]:
USE ROLE {{my_analyst_role}};
USE WAREHOUSE PLATPLUS_WH;

-- This works: {{my_analyst_role}} has SELECT on ORDERS
SELECT COUNT(*) AS order_count FROM {{my_db}}.RBAC_LAB.ORDERS;


> :white_check_mark: **Why this worked:** In Section 2, you granted `{{my_analyst_role}}` the full access chain — `USAGE` on the database, `USAGE` on the schema, and `SELECT` on ORDERS. All three links are present, so the query succeeds.


> :stop_sign: **The next cell will intentionally fail.** This is expected — it demonstrates what happens when a role is missing a privilege. When you see the error and a **"Fix Error" button appears, do NOT click it.** We want you to understand the error first and fix it yourself in the next section.

In [ ]:
-- This FAILS: {{my_analyst_role}} has NO grant on FEEDBACK
SELECT * FROM {{my_db}}.RBAC_LAB.FEEDBACK LIMIT 5;


> :mag_right: **Read the error message carefully.** You should see something like:
>
> `Object 'FEEDBACK' does not exist or not authorized. Your primary role {{my_analyst_role}} must have at least one privilege granted on TABLE FEEDBACK.`
>
> The table EXISTS. The data is THERE. Your role simply doesn't have a GRANT to read it. This is RBAC working exactly as designed — protecting data from unauthorized access.
>
> **The distinction:** "Does not exist or not authorized" does NOT necessarily mean the table is missing — it may well exist, but your current role can't see it. Snowflake deliberately uses this vague message to avoid leaking information about what objects exist to unauthorized roles.


## 4 · Grant the missing privilege — as the owner

You don't need SECURITYADMIN to fix this. You **own** these tables (you created them with `{{my_role}}`), so you can grant privileges on them.

**Why this matters (data analyst perspective):**
- **You may be both producer and consumer** — If you create tables (e.g., summary tables, materialized views), you own them and can control who else accesses them. This is common for analyst-built data products.
- **Understanding grants helps you request the right access** — When you need access to someone else's table, knowing the grant chain (database → schema → table) means you can tell your admin exactly what you need instead of a vague "I can't see the table."
- **Least privilege is your friend** — Granting only SELECT (not INSERT, UPDATE, DELETE) means collaborators can read your work without accidentally modifying it.

---

**Use cases where analysts grant access:**
| Scenario | What you grant | To whom |
|----------|---------------|--------|
| Sharing a summary table with your team | SELECT on the table | Team's role |
| Giving a dashboard read access to your view | SELECT on the view | Dashboard service role |
| Collaborating on a project schema | USAGE on schema + SELECT on specific tables | Colleague's role |
| Revoking access after a project ends | REVOKE SELECT | Former collaborator's role |

---

**Best practice — Least privilege:** Grant only the specific privilege needed (SELECT), on the specific object needed (FEEDBACK), to the specific role that needs it ({{my_analyst_role}}). Never grant more than required.

> :bulb: **Well-Architected principle:** In production, access changes should go through a formal request process (e.g., a Jira ticket that gets reviewed). In this lab, you're acting as both requester and approver to understand both sides.


In [ ]:
-- Switch back to {{my_role}} (the owner) to grant the missing privilege
USE ROLE {{my_role}};

-- {{my_analyst_role}} already has: WAREHOUSE (USAGE) + DATABASE (USAGE) + SCHEMA (USAGE)
-- The only missing link is SELECT on FEEDBACK
GRANT SELECT ON TABLE {{my_db}}.RBAC_LAB.FEEDBACK TO ROLE {{my_analyst_role}};


In [ ]:
USE ROLE {{my_analyst_role}};
SELECT * FROM {{my_db}}.RBAC_LAB.FEEDBACK LIMIT 5;


> :mag_right: **Same query, same data — the only thing that changed was a GRANT to the role.** That's RBAC in action.
>
> The pattern: 1) User hits access-denied → 2) Identify missing privilege → 3) Grant it to the role → 4) User retries — works.

---
## :robot_face: CoCo Assist

As a data analyst, CoCo isn't just for writing queries — it's your **access troubleshooting partner**. Instead of filing a ticket and waiting hours, you can ask CoCo to diagnose access issues in seconds. Here are three examples of how you'd use it in the real world.

> :warning: **How to get the most from this section:**
>
> 1. **Read CoCo's response carefully** — don't just glance at it. The explanation is where the learning happens.
> 2. **CoCo is interactive** — if something in the response doesn't make sense, ask a follow-up question. "What do you mean by 'inherited privilege'?" or "Can you show me an example?" The conversation continues until *you* understand.
> 3. **The lab effort is on you** — the prompts below are starting points. Your real learning comes from using the CoCo interface, reading the responses, and asking your own questions based on what you see.
> 4. **There's no single right answer** — CoCo may explain things differently each time. That's fine. Focus on understanding the concept, not memorizing the exact words.

---

**Example 1: Understand RBAC fundamentals**

A colleague asks: *"What's the difference between granting a privilege to a role vs directly to a user?"*

| Term | What it means |
|---|---|
| `GRANT ... TO ROLE` | Assigns a privilege to a role — anyone with that role inherits it |
| `GRANT ... TO USER` | Assigns directly to one person (UBAC) — doesn't scale |
| RBAC (Role-Based) | Standard approach — grant to roles, assign users to roles |
| UBAC (User-Based) | Direct-to-user — useful for private dev, not recommended for production |

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Explain the difference between granting a privilege to a role vs a user in Snowflake. When should I use each approach?</pre>

*Then follow up with:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Give me a real-world scenario where granting directly to a user would be appropriate.</pre>

---

**Example 2: Check your own privileges**

You want to know what access your current role has on the tables you just queried.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">What privileges does {{my_analyst_role}} have on the ORDERS and FEEDBACK tables in {{my_db}}.RBAC_LAB? Show me the grants.</pre>

*Then follow up with:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Why does {{my_analyst_role}} have SELECT but not INSERT? What would happen if I tried to INSERT?</pre>

This is the self-diagnosis skill — instead of guessing why a query failed, you can verify exactly what grants exist (or are missing).

---

**Example 3: Investigate past failures in query history**

Query history logs every query that ran — including failures. To troubleshoot something that *just* failed, use the real-time `INFORMATION_SCHEMA.QUERY_HISTORY` table function — it has no reporting latency, so a failure from moments ago shows up immediately (its `EXECUTION_STATUS` for a failed query is `FAILED_WITH_ERROR`). You can ask CoCo to find recent access-denied errors and explain why they happened.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Using the real-time INFORMATION_SCHEMA.QUERY_HISTORY table function, check for any queries against the FEEDBACK table in {{my_db}}.RBAC_LAB that failed with an error (EXECUTION_STATUS = 'FAILED_WITH_ERROR') in the last hour. Show me the error messages and explain why they failed.</pre>

*Then follow up with:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">What grant would have prevented that failure?</pre>

Query history records the SQL text, user, role, status, and error message for every query executed in the account. Two sources exist: the real-time `INFORMATION_SCHEMA.QUERY_HISTORY` table function used above (best for troubleshooting something that just happened), and the `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY` view (a 365-day history with up to ~2 hours of latency — better for audits and access reviews). It's invaluable for troubleshooting — you can see exactly what was attempted, by whom, and what went wrong.

> :bulb: **Two things to know about the table function.** It only reaches back **7 days** — query a wider window and it returns an error, so use the `ACCOUNT_USAGE` view for older history. And it returns **only your own queries** unless your role holds `MONITOR`/`OPERATE` on the warehouse (or is `ACCOUNTADMIN`) — a reminder that even query visibility follows RBAC. Here, the failed query was yours, so it shows up regardless of which role is active.

---

> :bulb: **These are just examples.** CoCo can help with any access-related question: "Why can't ROLE_X see this view?", "What's the minimum set of grants for a dashboard to read my table?", "Show me all roles that have access to the SALES schema." The point is that access troubleshooting doesn't have to mean memorizing SHOW GRANTS syntax — you can ask in plain English and follow up until it clicks.


---
## 5 · Your angle — Data Analyst: What can I see?

Before starting any analysis, know **what data is available to your role**. `SHOW GRANTS TO ROLE` lists every privilege your role holds.

**Why this matters (data analyst perspective):**
- **Scope your work before you start** — Nothing wastes more time than building a dashboard only to discover halfway through that you can't access a key table. Check your grants first.
- **Self-service access auditing** — Instead of asking an admin "what can I see?", you can answer that yourself with one command.
- **Understand your boundaries** — Knowing what you *can't* access is just as important. It tells you what to request, and from whom.

**This is the manual way.** In the CoCo Assist section above, you saw how CoCo can answer these questions conversationally. The SQL below does the same thing directly — it's important to understand both approaches:
- **CoCo** → faster for ad-hoc questions, explains results in plain English, good for troubleshooting
- **SHOW GRANTS SQL** → scriptable, repeatable, useful in automated checks or documentation

**Use cases:**
| Scenario | What you run | What it tells you |
|----------|-------------|-------------------|
| Starting a new project | `SHOW GRANTS TO ROLE <my_role>` | Full inventory of what data you can access |
| Debugging an access error | `SHOW GRANTS ON TABLE <table>` | Who has what privilege on that specific object |
| Preparing an access request | `SHOW GRANTS TO ROLE <my_role>` filtered for a schema | What you have vs. what you need — so you can request the gap |

In [ ]:
USE ROLE {{my_analyst_role}};
SHOW GRANTS TO ROLE {{my_analyst_role}};


> :mag_right: **Review.** You should see the complete grant chain for `{{my_analyst_role}}`:
> - USAGE on **warehouse** (PLATPLUS_WH — provides compute)
> - USAGE on **database** (your database)
> - USAGE on **schema** (RBAC_LAB)
> - SELECT on **ORDERS** and **FEEDBACK** (the two tables)
>
> This is the full chain we described in Section 2: warehouse → database → schema → table. All links are present, which is why both queries now succeed.
>
> **Analyst tip:** Run `SHOW GRANTS TO ROLE` at the start of any new project — it shows your complete access inventory. Faster than trial-and-error.


---
## 6 · Extend: Secondary roles

You can activate **secondary roles** to combine privileges from multiple roles in one session:

```sql
USE SECONDARY ROLES ALL;
```

**Default behaviour:** In this lab environment (and which is not a Snowflake default) Secondary roles are **OFF** (`NONE`) when a new session starts. You only get your primary role's privileges unless you explicitly activate secondary roles. This is a deliberate security design for these labs — access is opt-in, not automatic.

Options:
| Command | What it does |
|---------|-------------|
| `USE SECONDARY ROLES ALL` | Activates all roles granted to you |
| `USE SECONDARY ROLES NONE` | Deactivates all secondary roles (back to primary only — the default) |
| `USE SECONDARY ROLES (ROLE_A, ROLE_B)` | Activates only specific named roles |

**What `ALL` means:** Activates **every role that has been granted to you** as secondary roles. Your effective access becomes the union of your primary role + all your other granted roles combined. If you have 3 roles granted to you, `ALL` activates all of them simultaneously.

> :warning: **The one exception is `CREATE`.** Secondary roles contribute **all** the privileges of your other roles — reads *and* writes (`SELECT`, `INSERT`, `UPDATE`, `DELETE`, …) — **except `CREATE`**. Any new object you create is still owned by your **primary** (active) role, never a secondary one. So `ALL` expands far more than just your read access.

**Why this matters (data analyst perspective):**
- **Cross-project access** — You might have one role for marketing data and another for finance data. Secondary roles let you join tables from both without switching back and forth.
- **Avoiding "role gymnastics"** — Without secondary roles, analysts constantly switch roles to access different datasets. This is error-prone and frustrating.
- **Secondary roles expand more than reads** — you gain **all** privileges of your other granted roles (`SELECT`, `INSERT`, `UPDATE`, …) **except `CREATE`**. Your *primary* role still determines ownership of anything you create.

**Use cases:**
| Scenario | Primary role | Secondary roles provide |
|----------|-------------|------------------------|
| Cross-department reporting | MARKETING_ANALYST | Read access to FINANCE tables |
| Building a unified dashboard | ANALYTICS_RL | Read across multiple schemas |
| Investigating data lineage | YOUR_TEAM_ROLE | Read access to upstream tables |

> :bulb: **When to use:** When you need data across multiple schemas granted to different roles. Instead of switching roles, activate them all. If you find yourself constantly switching roles, ask your admin about secondary roles or a unified reporting role.

**Let's prove it works.** We'll revoke the SELECT grant from Section 4, show the query fails, then activate secondary roles to make it work again — without any new grants.

But first, let's run a SQL query to show how you can see your secondary role status, and confirm this has not been enabled.

In [ ]:
SELECT CURRENT_SECONDARY_ROLES();


#### Step 1: Revoke the SELECT we granted earlier

To demonstrate secondary roles, we first need to create a situation where `{{my_analyst_role}}` can't access FEEDBACK again. We'll **revoke** the SELECT grant we added in Section 4.

The cell also sets `USE SECONDARY ROLES NONE` — this ensures we're starting from a clean slate. If you've run cells out of order or are re-running the lab, secondary roles from a previous attempt might still be active (they persist for the session). Setting NONE guarantees the lab works correctly regardless of session history.

**Why this matters (data analyst perspective):**
- **REVOKE is the inverse of GRANT** — access can be removed just as easily as it's given. If a project ends or a team member changes roles, privileges get revoked.
- **Understanding revoke helps you understand temporary access** — In many organizations, access is granted for a project duration and revoked afterwards. This is normal, not punitive.


In [ ]:
-- Switch to the owner role, revoke SELECT, and ensure secondary roles are off
USE ROLE {{my_role}};
REVOKE SELECT ON TABLE {{my_db}}.RBAC_LAB.FEEDBACK FROM ROLE {{my_analyst_role}};

-- Now switch to {{my_analyst_role}} with NO secondary roles (clean slate)
USE ROLE {{my_analyst_role}};
USE SECONDARY ROLES NONE;


#### Step 2: Confirm the access is gone

Now switch to `{{my_analyst_role}}` and try to query FEEDBACK. It should fail — we're back to the "wall" from Section 3.

> :stop_sign: **This cell will intentionally fail** — that's the point. Don't click Fix Error.


In [ ]:
-- Switch to {{my_analyst_role}} and try to query FEEDBACK (should fail)
USE ROLE {{my_analyst_role}};
SELECT * FROM {{my_db}}.RBAC_LAB.FEEDBACK LIMIT 3;


> :white_check_mark: **Confirmed:** `{{my_analyst_role}}` can no longer see FEEDBACK — the revoke worked. Now let's fix this *without* granting the privilege back — using secondary roles instead.


#### Step 3: Activate secondary roles — the "unlock"

Your user has **both** `{{my_analyst_role}}` and `{{my_role}}` granted to it. Right now, `{{my_analyst_role}}` is your primary role (which can't see FEEDBACK). But `{{my_role}}` *owns* FEEDBACK — so it definitely has access.

By activating secondary roles, you combine the privileges of ALL your granted roles in one session — **every privilege except `CREATE`**:

```
Primary role ({{my_analyst_role}}):   WAREHOUSE + DATABASE + SCHEMA + SELECT on ORDERS only
Secondary role ({{my_role}}): OWNERSHIP on both tables (includes all privileges)
──────────────────────────────────────────────────────────────────────────────────
Combined effective access:   Everything from both roles — including FEEDBACK
```

**Why this matters (data analyst perspective):**
- **No need to switch roles** — You stay as {{my_analyst_role}} (your "identity" for audit purposes) but gain read access from your other roles
- **No need to ask an admin for a new grant** — If you already have another role that has the access, secondary roles unlock it immediately
- **Audit trail is preserved** — Queries still log under your primary role, so access auditing remains clear


In [ ]:
-- Activate ALL secondary roles — combines privileges from every role granted to you
USE ROLE {{my_analyst_role}};
USE SECONDARY ROLES ALL;

-- Verify what's active
SELECT CURRENT_ROLE() AS primary_role, CURRENT_SECONDARY_ROLES() AS secondary_roles;


> :mag_right: **What you should see:** `primary_role` = {{my_analyst_role}}, `secondary_roles` includes {{my_role}}. This means you now have the combined access of both roles.


#### Step 4: Re-run the query — it works now

Same role ({{my_analyst_role}}), same query, same table — but now it succeeds because `{{my_role}}`'s ownership privileges flow through as a secondary role.


In [ ]:
-- Same query that failed moments ago — now works via secondary roles
SELECT * FROM {{my_db}}.RBAC_LAB.FEEDBACK LIMIT 3;


> :white_check_mark: **The "aha" moment:** Nothing changed about `{{my_analyst_role}}`'s grants. No admin was involved. You simply activated a role you already had — and gained its privileges alongside your primary role.
>
> **This is the power of secondary roles for analysts:** When you have multiple roles for different projects or data domains, you don't need to switch between them or request new grants. Activate them all and work across everything you're authorized to see.


#### Step 5: Clean up — restore to true original state

This lab is one of three RBAC persona labs (Data Analyst, Data Engineer, Platform Admin). They share the same database and objects, and each lab expects to start from the same baseline. The cleanup below restores the environment to the **true original state** from Section 2:
- `{{my_analyst_role}}` has ORDERS access but **NOT** FEEDBACK
- Secondary roles are deactivated
- All tables and schemas remain intact

This ensures that if you (or another student) run a different persona lab next, it starts from the expected baseline — with the FEEDBACK gap in place for that lab to demonstrate its own scenarios.

**What the cleanup does:**
- Revokes SELECT on FEEDBACK from {{my_analyst_role}} (back to the deliberate gap)
- Keeps SELECT on ORDERS ({{my_analyst_role}} still has partial access)
- Deactivates secondary roles (back to default NONE)
- Leaves all tables, schemas, and roles intact (non-destructive)


In [ ]:
-- Clean up: restore to true original state (Section 2 baseline)
USE ROLE {{my_role}};

-- Ensure the full chain is in place for ORDERS (the "working" table)
GRANT USAGE  ON DATABASE {{my_db}}                    TO ROLE {{my_analyst_role}};
GRANT USAGE  ON SCHEMA   {{my_db}}.RBAC_LAB           TO ROLE {{my_analyst_role}};
GRANT SELECT ON TABLE    {{my_db}}.RBAC_LAB.ORDERS    TO ROLE {{my_analyst_role}};

-- Revoke FEEDBACK access (restore the deliberate gap for other persona labs)
REVOKE SELECT ON TABLE {{my_db}}.RBAC_LAB.FEEDBACK FROM ROLE {{my_analyst_role}};

-- Deactivate secondary roles (back to default)
USE ROLE {{my_analyst_role}};
USE SECONDARY ROLES NONE;


---
## Verify Your Work

The cleanup restored the environment to the **true original state** from Section 2: {{my_analyst_role}} has ORDERS but NOT FEEDBACK. The cells below prove both sides:
1. ORDERS query succeeds (access is intact)
2. FEEDBACK query fails (the deliberate gap is back)

| Check | Expected Result |
|-------|----------------|
| ORDERS query | Succeeds — returns row count |
| FEEDBACK query | **Fails** — access denied (this is correct!) |
| Secondary roles deactivated | Session is clean |
| All tables/schemas intact | Nothing was dropped |


In [ ]:
-- Final verification Part 1: ORDERS is accessible (should succeed)
USE ROLE {{my_analyst_role}};
SELECT COUNT(*) AS orders_count FROM {{my_db}}.RBAC_LAB.ORDERS;


> :stop_sign: **The next cell will intentionally fail** — this proves the cleanup worked and FEEDBACK access has been revoked back to the original state.

In [ ]:
-- Final verification Part 2: FEEDBACK is NOT accessible (should fail — proves original state restored)
SELECT COUNT(*) AS feedback_count FROM {{my_db}}.RBAC_LAB.FEEDBACK;


---
### Check your understanding 🧠
A few questions on this lab's key themes. Run each quiz cell below, choose an answer, and review the feedback.

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q4")

---
### :snowflake: Key Takeaways

- Privileges go to **roles**, roles go to users — that's RBAC
- The full grant chain: **Warehouse → Database → Schema → Table** — missing any link = access denied
- "Does not exist or not authorized" does NOT necessarily mean the object is missing — it may exist, but your role can't see it
- An object **owner** can grant and revoke privileges on what they own
- `REVOKE` is the inverse of `GRANT` — access can be removed as easily as it's given
- `SHOW GRANTS TO ROLE` is your self-diagnosis tool — run it before starting any project
- `USE SECONDARY ROLES ALL` combines privileges from all your granted roles (default is NONE)
- **CoCo is your access troubleshooting partner** — ask it to check grants, investigate failures, or explain errors

**The Other Personas:**
- The **Data Engineer** notebook covers future grants and schema-wide automation
- The **Platform Admin** notebook covers auditing, revoking, and governance

> :bulb: Each persona lab is independent — you can run them in any order. The cleanup cell restores the environment to baseline between labs.

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.